8mins 30.7 secs

## Libraries

In [24]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.model_selection import ParameterGrid

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

In [25]:
print("Torch version:", torch.__version__)
print("CUDA (in torch):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

Torch version: 2.10.0.dev20251203+cu128
CUDA (in torch): 12.8
cuda.is_available: True
device count: 1
device: NVIDIA GeForce RTX 5070 Ti
capability: (12, 0)


## Config

In [26]:
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TUNE_START_DATE = pd.Timestamp("2007-04-01")
TUNE_END_DATE   = pd.Timestamp("2022-03-31")

ROLLING_TRAIN_WINDOW = 90   # 7.5 years
ROLLING_VAL_WINDOW   = 18   # 1.5 years

param_grid = list(ParameterGrid({
    "WINDOW":      [12, 18],
    "HIDDEN_DIM":  [32, 64, 128],
    "DROPOUT":     [0.0, 0.3],
    "LR":          [1e-3, 5e-4],
    "WEIGHT_DECAY":[0.0, 1e-4],
    "GATE_INIT":   [0.8],   # initial preference for distance graph
    "K_DIST":      [8],
    "SIGMA_KM":    [None, 60.0],  # None=binary weights; 60km exponential weights
    "K_CORR":      [8],
}))

BATCH_SIZE    = 32
MAX_EPOCHS    = 80
PATIENCE      = 6
MAX_GRAD_NORM = 5.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ----------------------------
# Feature lists
# ----------------------------
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

base_feature_cols = continuous_cols + categorical_cols

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Using device: cuda


## Metric functions

In [27]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)


## Load data

In [28]:
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

mask_tune = (df[TIME_COL] >= TUNE_START_DATE) & (df[TIME_COL] <= TUNE_END_DATE)
df = df.loc[mask_tune].copy()

df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)

print(f"Tuning period: {dates[0].date()} → {dates[-1].date()}")
print("Total months:", T_total)
print("Number of LAs:", N)

full_index = pd.MultiIndex.from_product([dates, la_order], names=[TIME_COL, ENTITY_COL])
feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# Forward-fill FEATURES only (never bfill). Leave TARGET missing if missing.
df_panel[feature_cols] = (
    df_panel[feature_cols]
      .groupby(level=ENTITY_COL)
      .ffill()
)

# Add price lags (do not bfill)
df_panel["price_lag1"]  = df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
df_panel["price_lag12"] = df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)

# Forward-fill lag features only
df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
      .groupby(level=ENTITY_COL)
      .ffill()
)

lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols
F = len(feature_cols)

X_all = (
    df_panel[feature_cols]
      .to_numpy(dtype=np.float32)
      .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
      .to_numpy(dtype=np.float32)
      .reshape(T_total, N)
)

y_all_orig = y_all.copy()

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

Tuning period: 2007-04-01 → 2022-03-01
Total months: 180
Number of LAs: 294
X_all shape: (180, 294, 36)
y_all shape: (180, 294)


## Rolling origin folds (time index space)

In [29]:
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > T_total:
        break

    train_start_idx = train_end_idx - ROLLING_TRAIN_WINDOW
    fold_specs.append((
        train_start_idx, train_end_idx, val_start_idx, val_end_idx,
        dates[train_start_idx], dates[train_end_idx - 1],
        dates[val_start_idx], dates[val_end_idx - 1],
    ))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")
for i, (tr_s, tr_e, v_s, v_e, ts, te, vs, ve) in enumerate(fold_specs, start=1):
    print(f"  Fold {i}: Train {ts:%Y-%m}–{te:%Y-%m}, Val {vs:%Y-%m}–{ve:%Y-%m}")


Number of folds: 5
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
  Fold 2: Train 2008-10–2016-03, Val 2016-04–2017-09
  Fold 3: Train 2010-04–2017-09, Val 2017-10–2019-03
  Fold 4: Train 2011-10–2019-03, Val 2019-04–2020-09
  Fold 5: Train 2013-04–2020-09, Val 2020-10–2022-03


## Adjacency builders

In [30]:
def build_distance_knn_Ahat_from_panel(
    df_panel: pd.DataFrame,
    dates: pd.Index,
    la_order: list,
    k_dist: int = 8,
    sigma_km: float | None = None,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    Uses centroid_x/centroid_y in EPSG:27700 (meters).
    Builds symmetric KNN adjacency; weights are either binary or exp(-d_km/sigma_km).
    Adds self-loops and returns degree-normalized A_hat (float32).
    """
    first_date = dates[0]
    cent = df_panel.loc[(first_date, la_order), ["centroid_x", "centroid_y"]].to_numpy(dtype=np.float32)

    nbrs = NearestNeighbors(n_neighbors=k_dist + 1, algorithm="auto").fit(cent)
    dists_m, idx = nbrs.kneighbors(cent)

    Nloc = len(la_order)
    A = np.zeros((Nloc, Nloc), dtype=np.float32)

    for i in range(Nloc):
        for n in range(1, k_dist + 1):  # skip self
            j = int(idx[i, n])
            d_km = float(dists_m[i, n] / 1000.0)

            if sigma_km is None:
                w = 1.0
            else:
                w = float(np.exp(-d_km / (sigma_km + eps)))

            if w > A[i, j]:
                A[i, j] = w
            if w > A[j, i]:
                A[j, i] = w

    A = A + np.eye(Nloc, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    A_hat = (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)
    return A_hat

def build_corr_knn_Ahat_train_only(
    y_train_TN: np.ndarray,
    k_corr: int = 8,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    y_train_TN: [T_train, N] may contain NaN.
    Fills NaNs with per-node TRAIN mean, then builds abs(corr) KNN graph.
    Adds self-loops and returns degree-normalized A_hat (float32).
    """
    y = y_train_TN.copy()
    col_means = np.nanmean(y, axis=0)
    inds = np.where(np.isnan(y))
    if inds[0].size > 0:
        y[inds] = np.take(col_means, inds[1])

    corr = np.corrcoef(y.T)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(corr, 0.0)

    Nloc = corr.shape[0]
    A = np.zeros((Nloc, Nloc), dtype=np.float32)

    for i in range(Nloc):
        nbr_idx = np.argsort(-np.abs(corr[i]))[:k_corr]
        for j in nbr_idx:
            A[i, j] = 1.0
            A[j, i] = 1.0

    A = A + np.eye(Nloc, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    A_hat = (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)
    return A_hat

## Dataset class (window will vary per config)

In [31]:
class SpatioTemporalDataset(Dataset):
    """
    Each sample is (X_seq, y_t):
      - X_seq: [window, N, F]
      - y_t: [N]
    """
    def __init__(self, X, y, window):
        self.X = X
        self.y = y
        self.window = window
        self.T, self.N, self.F = X.shape
        self.indices = list(range(window, self.T))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        X_seq = self.X[t - self.window:t]  # [window, N, F]
        y_t   = self.y[t]                  # [N]
        return torch.tensor(X_seq, dtype=torch.float32), torch.tensor(y_t, dtype=torch.float32)


## Model

In [32]:
class GraphConv2Gated(nn.Module):
    """
    Two-channel graph convolution with shared feature transform and a learnable gate:
      out = g * linear(A_dist X) + (1-g) * linear(A_corr X)
    Gate is a learnable scalar (logit parameterization).
    """
    def __init__(self, in_feats, out_feats, gate_init=0.8):
        super().__init__()
        self.linear = nn.Linear(in_feats, out_feats)

        # gate_init in (0,1)
        gate_init = float(np.clip(gate_init, 1e-4, 1 - 1e-4))
        init_logit = np.log(gate_init / (1.0 - gate_init))
        self.gate_logit = nn.Parameter(torch.tensor(init_logit, dtype=torch.float32))

    def forward(self, X, A_dist, A_corr):
        # X: [B, N, F], A_*: [N, N]
        AX_dist = torch.einsum("ij,bjf->bif", A_dist, X)
        AX_corr = torch.einsum("ij,bjf->bif", A_corr, X)

        out_dist = self.linear(AX_dist)
        out_corr = self.linear(AX_corr)

        g = torch.sigmoid(self.gate_logit)  # scalar
        return g * out_dist + (1.0 - g) * out_corr

    def gate_value(self) -> float:
        return float(torch.sigmoid(self.gate_logit).detach().cpu().item())

class TGCNCell2(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0, gate_init=0.8):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gc_zr = GraphConv2Gated(in_feats + hidden_dim, 2 * hidden_dim, gate_init=gate_init)
        self.gc_h  = GraphConv2Gated(in_feats + hidden_dim, hidden_dim,     gate_init=gate_init)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X_t, H_prev, A_dist, A_corr):
        if H_prev is None:
            H_prev = torch.zeros(X_t.size(0), X_t.size(1), self.hidden_dim, device=X_t.device)

        XH = torch.cat([X_t, H_prev], dim=-1)

        ZR = torch.sigmoid(self.gc_zr(XH, A_dist, A_corr))
        Z, R = torch.chunk(ZR, 2, dim=-1)

        XH_candidate = torch.cat([X_t, R * H_prev], dim=-1)
        H_tilde = torch.tanh(self.gc_h(XH_candidate, A_dist, A_corr))

        H_new = (1 - Z) * H_prev + Z * H_tilde
        H_new = self.dropout(H_new)
        return H_new

class TGCN2(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0, gate_init=0.8):
        super().__init__()
        self.cell = TGCNCell2(in_feats, hidden_dim, dropout=dropout, gate_init=gate_init)
        self.out  = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_dist, A_corr):
        # X_seq: [B, T, N, F]
        H = None
        for t in range(X_seq.size(1)):
            H = self.cell(X_seq[:, t], H, A_dist, A_corr)
        return self.out(H).squeeze(-1)  # [B, N]


## Train and Eval with early stopping

In [ ]:
results = []

for cfg_id, params in enumerate(param_grid, start=1):
    WINDOW      = params["WINDOW"]
    HIDDEN_DIM  = params["HIDDEN_DIM"]
    DROPOUT     = params["DROPOUT"]
    LR          = params["LR"]
    WD          = params["WEIGHT_DECAY"]
    GATE_INIT   = params["GATE_INIT"]
    K_DIST      = params["K_DIST"]
    SIGMA_KM    = params["SIGMA_KM"]
    K_CORR      = params["K_CORR"]

    print(f"\n=== Config {cfg_id}/{len(param_grid)} ===")
    print(params)

    # Build distance adjacency once per config (depends on K_DIST, SIGMA_KM)
    A_hat_dist_np = build_distance_knn_Ahat_from_panel(
        df_panel=df_panel,
        dates=dates,
        la_order=la_order,
        k_dist=K_DIST,
        sigma_km=SIGMA_KM,
    )
    A_dist = torch.tensor(A_hat_dist_np, dtype=torch.float32, device=DEVICE)

    fold_mae_list, fold_rmse_list, fold_smape_list, fold_mase_list = [], [], [], []
    fold_count = 0

    for fold_no, (train_start_idx, train_end_idx,
                  val_start_idx, val_end_idx,
                  train_start_date, train_end_date,
                  val_start_date, val_end_date) in enumerate(fold_specs, start=1):

        print(f"  Fold {fold_no}: Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
              f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}")

        # --------------------------
        # Split
        # --------------------------
        X_train = X_all[train_start_idx:train_end_idx]  # [Ttr,N,F]
        y_train = y_all[train_start_idx:train_end_idx]  # [Ttr,N]
        X_val   = X_all[val_start_idx:val_end_idx]      # [Tva,N,F]
        y_val   = y_all[val_start_idx:val_end_idx]      # [Tva,N]

        if X_train.shape[0] < WINDOW + 1 or X_val.shape[0] < 1:
            print("    (skip fold: not enough time steps)")
            continue

        # --------------------------
        # Fold-specific correlation adjacency (TRAIN ONLY)
        # --------------------------
        A_hat_corr_np = build_corr_knn_Ahat_train_only(y_train, k_corr=K_CORR)
        A_corr = torch.tensor(A_hat_corr_np, dtype=torch.float32, device=DEVICE)

        # --------------------------
        # Feature filling: last-resort fill using TRAIN means (FEATURES only)
        # --------------------------
        if np.isnan(X_train).any():
            train_feat_means = np.nanmean(X_train.reshape(-1, F), axis=0)
            X_train = np.where(np.isnan(X_train), train_feat_means[None, None, :], X_train)

        if np.isnan(X_val).any():
            train_feat_means = np.nanmean(X_train.reshape(-1, F), axis=0)
            X_val = np.where(np.isnan(X_val), train_feat_means[None, None, :], X_val)

        # --------------------------
        # Scaling (fit on TRAIN ONLY)
        # --------------------------
        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train.reshape(-1, F)).reshape(X_train.shape)
        X_val_scaled   = x_scaler.transform(X_val.reshape(-1, F)).reshape(X_val.shape)

        # y scaler on observed TRAIN targets only
        y_train_flat = y_train.reshape(-1, 1)
        y_train_obs  = y_train_flat[~np.isnan(y_train_flat).ravel()].reshape(-1, 1)
        if y_train_obs.shape[0] < 10:
            print("    (skip fold: too few observed train targets)")
            continue

        y_scaler = RobustScaler()
        y_scaler.fit(y_train_obs)
        y_scale_factor = float(y_scaler.scale_[0])

        def transform_y_keep_nan(y_arr: np.ndarray) -> np.ndarray:
            y2 = y_arr.reshape(-1, 1).copy()
            mask = ~np.isnan(y2).ravel()
            out = np.full_like(y2, np.nan, dtype=np.float32)
            out[mask] = y_scaler.transform(y2[mask]).astype(np.float32)
            return out.reshape(y_arr.shape)

        y_train_scaled = transform_y_keep_nan(y_train).astype(np.float32)
        y_val_scaled   = transform_y_keep_nan(y_val).astype(np.float32)

        # --------------------------
        # Build VAL with TRAIN tail context
        # --------------------------
        X_context = np.concatenate([X_train_scaled[-WINDOW:], X_val_scaled], axis=0)  # [WINDOW+Tva, N, F]
        y_context = np.concatenate([y_train_scaled[-WINDOW:], y_val_scaled], axis=0)  # [WINDOW+Tva, N]

        train_ds = SpatioTemporalDataset(X_train_scaled, y_train_scaled, window=WINDOW)
        val_ds   = SpatioTemporalDataset(X_context,      y_context,      window=WINDOW)

        if len(train_ds) == 0 or len(val_ds) == 0:
            print("    (skip fold: empty dataset after windowing)")
            continue

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        # --------------------------
        # Model
        # --------------------------
        model = TGCN2(in_feats=F, hidden_dim=HIDDEN_DIM, dropout=DROPOUT, gate_init=GATE_INIT).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        loss_fn = nn.MSELoss()

        best_val_mse = np.inf
        best_epoch = -1
        epochs_no_improve = 0
        best_state = None

        # --------------------------
        # Training with early stopping
        # Strategy for missing y: skip batches with any NaN in y_t
        # If you have a lot of missing targets, switch to a masked loss instead.
        # --------------------------
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []

            for X_seq, y_t in train_loader:
                if torch.isnan(y_t).any():
                    continue

                X_seq = X_seq.to(DEVICE)  # [B,T,N,F]
                y_t   = y_t.to(DEVICE)    # [B,N]

                optimizer.zero_grad()
                y_hat = model(X_seq, A_dist, A_corr)
                loss  = loss_fn(y_hat, y_t)

                if not torch.isfinite(loss):
                    train_losses = []
                    break

                loss.backward()
                clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
                optimizer.step()
                train_losses.append(loss.item())

            if len(train_losses) == 0:
                print("    ⚠ No valid training batches (targets missing?). Skipping fold.")
                break

            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_seq, y_t in val_loader:
                    if torch.isnan(y_t).any():
                        continue
                    X_seq = X_seq.to(DEVICE)
                    y_t   = y_t.to(DEVICE)
                    y_hat = model(X_seq, A_dist, A_corr)
                    vloss = loss_fn(y_hat, y_t)
                    if torch.isfinite(vloss):
                        val_losses.append(vloss.item())

            if len(val_losses) == 0:
                print("    ⚠ No valid validation batches. Skipping fold.")
                break

            val_mse = float(np.mean(val_losses))
            val_rmse_orig = float(np.sqrt(val_mse) * y_scale_factor)

            g_zr = model.cell.gc_zr.gate_value()
            g_h  = model.cell.gc_h.gate_value()

            print(f"    Epoch {epoch:03d} | train MSE={np.mean(train_losses):.4f} | "
                  f"val MSE={val_mse:.4f} | val RMSE(£)≈{val_rmse_orig:,.1f} | "
                  f"gates(zr={g_zr:.3f}, h={g_h:.3f})")

            if val_mse + 1e-6 < best_val_mse:
                best_val_mse = val_mse
                best_epoch   = epoch
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    print(f"    Early stopping at epoch {epoch}")
                    break

        if best_epoch == -1 or best_state is None:
            print("    ❌ Fold failed (no valid epoch).")
            continue

        model.load_state_dict(best_state)

        # --------------------------
        # Final validation predictions (only val months via context dataset)
        # --------------------------
        model.eval()
        y_true_scaled_list = []
        y_pred_scaled_list = []

        with torch.no_grad():
            for X_seq, y_t in val_loader:
                if torch.isnan(y_t).any():
                    continue
                X_seq = X_seq.to(DEVICE)
                y_t   = y_t.to(DEVICE)
                y_hat = model(X_seq, A_dist, A_corr)

                y_true_scaled_list.append(y_t.cpu().numpy())   # [B,N]
                y_pred_scaled_list.append(y_hat.cpu().numpy()) # [B,N]

        if len(y_true_scaled_list) == 0:
            print("    ❌ No valid validation predictions (targets missing?).")
            continue

        y_true_val_scaled = np.concatenate(y_true_scaled_list, axis=0).reshape(-1, 1)
        y_pred_val_scaled = np.concatenate(y_pred_scaled_list, axis=0).reshape(-1, 1)

        y_true_val_orig = y_scaler.inverse_transform(y_true_val_scaled).ravel()
        y_pred_val_orig = y_scaler.inverse_transform(y_pred_val_scaled).ravel()

        y_train_fold_orig = y_all_orig[train_start_idx:train_end_idx].reshape(-1)
        y_train_fold_orig = y_train_fold_orig[~np.isnan(y_train_fold_orig)]

        fold_mae  = mae(y_true_val_orig, y_pred_val_orig)
        fold_rmse = rmse(y_true_val_orig, y_pred_val_orig)
        fold_smape = smape(y_true_val_orig, y_pred_val_orig)
        fold_mase = mase(y_true_val_orig, y_pred_val_orig, y_train_fold_orig, m=12)

        g_zr = model.cell.gc_zr.gate_value()
        g_h  = model.cell.gc_h.gate_value()

        print(f"    Fold {fold_no} MAE(£)={fold_mae:,.1f}, RMSE(£)={fold_rmse:,.1f}, "
              f"sMAPE={fold_smape:.3f}%, MASE={fold_mase:.3f} | "
              f"final gates(zr={g_zr:.3f}, h={g_h:.3f})")

        fold_mae_list.append(fold_mae)
        fold_rmse_list.append(fold_rmse)
        fold_smape_list.append(fold_smape)
        fold_mase_list.append(fold_mase)
        fold_count += 1

    if fold_count == 0:
        print("  ❌ No valid folds for this config. Skipping.")
        continue

    cfg_result = {
        "model_type": "TGCN_C1_GATED",
        "WINDOW": WINDOW,
        "HIDDEN_DIM": HIDDEN_DIM,
        "DROPOUT": DROPOUT,
        "LR": LR,
        "WEIGHT_DECAY": WD,
        "GATE_INIT": GATE_INIT,
        "K_DIST": K_DIST,
        "SIGMA_KM": SIGMA_KM,
        "K_CORR": K_CORR,
        "folds_used": fold_count,

        "MAE_mean":   float(np.mean(fold_mae_list)),
        "MAE_std":    float(np.std(fold_mae_list)),
        "RMSE_mean":  float(np.mean(fold_rmse_list)),
        "RMSE_std":   float(np.std(fold_rmse_list)),
        "sMAPE_mean": float(np.mean(fold_smape_list)),
        "sMAPE_std":  float(np.std(fold_smape_list)),
        "MASE_mean":  float(np.mean(fold_mase_list)),
        "MASE_std":   float(np.std(fold_mase_list)),
    }
    results.append(cfg_result)



=== Config 1/96 ===
{'DROPOUT': 0.0, 'GATE_INIT': 0.8, 'HIDDEN_DIM': 32, 'K_CORR': 8, 'K_DIST': 8, 'LR': 0.001, 'SIGMA_KM': None, 'WEIGHT_DECAY': 0.0, 'WINDOW': 12}
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
    Epoch 001 | train MSE=1.0961 | val MSE=2.3039 | val RMSE(£)≈154,245.3 | gates(zr=0.800, h=0.800)
    Epoch 002 | train MSE=0.8812 | val MSE=1.9731 | val RMSE(£)≈142,742.9 | gates(zr=0.800, h=0.799)
    Epoch 003 | train MSE=0.7518 | val MSE=1.6873 | val RMSE(£)≈132,002.2 | gates(zr=0.800, h=0.799)
    Epoch 004 | train MSE=0.6157 | val MSE=1.4446 | val RMSE(£)≈122,140.2 | gates(zr=0.799, h=0.798)
    Epoch 005 | train MSE=0.5100 | val MSE=1.2501 | val RMSE(£)≈113,620.7 | gates(zr=0.799, h=0.798)
    Epoch 006 | train MSE=0.4544 | val MSE=1.1023 | val RMSE(£)≈106,691.7 | gates(zr=0.798, h=0.797)
    Epoch 007 | train MSE=0.4152 | val MSE=0.9954 | val RMSE(£)≈101,386.2 | gates(zr=0.798, h=0.796)
    Epoch 008 | train MSE=0.3897 | val MSE=0.9234 | val RMSE(£)≈97,648.9 |

## Results

In [ ]:
results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("RMSE_mean").reset_index(drop=True)
    print("\n=== TOP CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===")
    print(results_df.head(10))
    out_path = "../../results/tgcn_c1_gated_rollingcv_results_fixed.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\nSaved tuning results to {out_path}")
else:
    print("\nNo successful configs to report.")


=== TOP T-GCN CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===
  model_type  WINDOW  HIDDEN_DIM  DROPOUT      LR  WEIGHT_DECAY  folds_used  \
0       TGCN      12         128      0.0  0.0010        0.0001           5   
1       TGCN      12         128      0.0  0.0010        0.0000           5   
2       TGCN      12          64      0.0  0.0010        0.0001           5   
3       TGCN      12         128      0.0  0.0005        0.0000           5   
4       TGCN      12          64      0.0  0.0010        0.0000           5   
5       TGCN      12         128      0.3  0.0010        0.0001           5   
6       TGCN      12         128      0.0  0.0005        0.0001           5   
7       TGCN      12         128      0.3  0.0010        0.0000           5   
8       TGCN      12          64      0.0  0.0005        0.0000           5   
9       TGCN      12          64      0.3  0.0010        0.0001           5   

       MAE_mean      MAE_std     RMSE_mean      RMSE_std  sMAPE_mean  \
0